<a href="https://colab.research.google.com/github/Swayam-dot-cmd/neural-information-retrieval/blob/main/notebooks/06_hybrid_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q beir sentence-transformers rank_bm25 scikit-learn

In [ ]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = "./datasets"
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id]["text"] for doc_id in doc_ids]

In [ ]:
tokenized_corpus = [doc.split() for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
dense_model = SentenceTransformer('BAAI/bge-small-en')

corpus_embeddings = dense_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

In [ ]:
def hybrid_retrieve(query, alpha=0.5, top_k=10):
    # BM25 scores
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # BGE scores
    query_embedding = dense_model.encode([query], convert_to_numpy=True)[0]
    dense_scores = cosine_similarity([query_embedding], corpus_embeddings)[0]

    # Normalize scores
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)
    dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-8)

    # Combine
    final_scores = alpha * bm25_scores + (1 - alpha) * dense_scores

    top_indices = np.argsort(final_scores)[::-1][:top_k]

    return [doc_ids[i] for i in top_indices]

In [ ]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)

    return len(retrieved_k & relevant_set) / len(relevant_set)

In [ ]:
recalls = []

for qid in queries:
    query = queries[qid]

    retrieved_docs = hybrid_retrieve(query, alpha=0.2, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

avg_recall = sum(recalls) / len(recalls)

print("Hybrid Recall@10:", avg_recall)

In [ ]:
for alpha in [0.2, 0.4, 0.5, 0.6, 0.8]:
    recalls = []

    for qid in queries:
        query = queries[qid]

        retrieved_docs = hybrid_retrieve(query, alpha=alpha, top_k=10)
        relevant_docs = list(qrels[qid].keys())

        if len(relevant_docs) == 0:
            continue

        recall = recall_at_k(retrieved_docs, relevant_docs, 10)
        recalls.append(recall)

    print(f"Alpha {alpha}: {sum(recalls)/len(recalls)}")